In [1]:
!ls MsceneSpeech

test  train


In [2]:
!ls MsceneSpeech/train

chat  news  qa	story


In [3]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [11]:
files = glob('MsceneSpeech/*/*/*/*.txt')
files = [f for f in files if 'pinyin' not in f]
len(files)

5108

In [14]:
!mkdir MsceneSpeech_audio

In [17]:
os.path.split(f)[0].replace('/', '_')

'MsceneSpeech_train_news_新闻稿四_出雲'

In [24]:
def loop(files):
    files, _ = files
    data = []
    for f in tqdm(files):
        try:
            with open(f) as fopen:
                t = fopen.read()
            t = t.strip()
            if len(t) < 2:
                continue
                
            f = f.replace('.txt', '.wav')

            audio_np, sr = sf.read(f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
    
            audio_filename = f.replace('/', '_').replace('.wav', '.mp3')
            audio_filename = os.path.join('MsceneSpeech_audio', audio_filename)
                
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': os.path.split(f)[0].replace('/', '_'),
            })
        except Exception as e:
            print(e)
            pass
    return data

In [25]:
data = loop((files[:10], 0))
data

100%|██████████| 10/10 [00:01<00:00,  7.26it/s]


[{'audio_filename': 'MsceneSpeech_audio/MsceneSpeech_train_news_新闻稿四_出雲_28.mp3',
  'text': '“坚持弘扬平等、互鉴、对话、包容的文明观，弘扬中华文明蕴含的全人类共同价值。',
  'speaker': 'MsceneSpeech_train_news_新闻稿四_出雲'},
 {'audio_filename': 'MsceneSpeech_audio/MsceneSpeech_train_news_新闻稿四_出雲_90.mp3',
  'text': '坚持系统保护、协同保护、特殊保护，以科学精神制定一部经得起历史和人民检验的好法律。',
  'speaker': 'MsceneSpeech_train_news_新闻稿四_出雲'},
 {'audio_filename': 'MsceneSpeech_audio/MsceneSpeech_train_news_新闻稿四_出雲_364.mp3',
  'text': '以习近平同志为核心的党中央深刻把握国际国内大局大势，坚持稳中求进工作总基调，',
  'speaker': 'MsceneSpeech_train_news_新闻稿四_出雲'},
 {'audio_filename': 'MsceneSpeech_audio/MsceneSpeech_train_news_新闻稿四_出雲_254.mp3',
  'text': '希望自治区党委和政府团结带领广大干部群众，同心协力，砥砺前进，扎实做好新疆各项工作。',
  'speaker': 'MsceneSpeech_train_news_新闻稿四_出雲'},
 {'audio_filename': 'MsceneSpeech_audio/MsceneSpeech_train_news_新闻稿四_出雲_125.mp3',
  'text': '他强调，要积极推进湿地建设，加强生物多样性保护，减少人类活动干扰，促进人与自然和谐共生，不断筑牢国家生态安全屏障。',
  'speaker': 'MsceneSpeech_train_news_新闻稿四_出雲'},
 {'audio_filename': 'MsceneSpeech_audio/MsceneSpeech_train_ne

In [27]:
data = multiprocessing(files, loop, cores = 20)

 42%|████▏     | 107/255 [00:13<00:16,  8.80it/s]

Error opening 'MsceneSpeech/train/story/故事讲述文本二_拨云观星/106(1).wav': System error.

 49%|████▉     | 125/255 [00:13<00:14,  8.83it/s]

 55%|█████▌    | 141/255 [00:19<00:15,  7.23it/s]

Error opening 'MsceneSpeech/train/story/故事讲述文本二_拨云观星/112(1).wav': System error.

 71%|███████▏  | 182/255 [00:19<00:07,  9.34it/s]

100%|██████████| 255/255 [00:36<00:00,  7.06it/s]


In [28]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'MsceneSpeech_audio/MsceneSpeech_train_news_新闻稿四_出雲_28.mp3',
 'text': '“坚持弘扬平等、互鉴、对话、包容的文明观，弘扬中华文明蕴含的全人类共同价值。',
 'speaker': 'MsceneSpeech_train_news_新闻稿四_出雲'}

In [29]:
dataset.push_to_hub('malaysia-ai/MsceneSpeech')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 279.60ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  496kB /  496kB, 1.24MB/s  
Processing Files (1 / 1): 100%|██████████|  496kB /  496kB,  827kB/s  
New Data Upload: 100%|██████████|  496kB /  496kB,  827kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.03 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/MsceneSpeech/commit/8f5f2021c5f170e2569e05f6666ad326ef663073', commit_message='Upload dataset', commit_description='', oid='8f5f2021c5f170e2569e05f6666ad326ef663073', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/MsceneSpeech', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/MsceneSpeech'), pr_revision=None, pr_num=None)

In [30]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'MsceneSpeech')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 281.04ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  496kB /  496kB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  2.76 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/0bc2b04d6c4658d2e47e40d9e932118d1739d9e7', commit_message='Upload dataset', commit_description='', oid='0bc2b04d6c4658d2e47e40d9e932118d1739d9e7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [32]:
audio_files = [d['audio_filename'] for d in data]

with open('MsceneSpeech-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [33]:
!zip -rq MsceneSpeech_audio_neucodec.zip MsceneSpeech_audio_neucodec

In [34]:
!zip -rq MsceneSpeech_audio.zip MsceneSpeech_audio

In [35]:
!hf upload malaysia-ai/Multilingual-TTS MsceneSpeech_audio_neucodec.zip --repo-type=dataset

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...Speech_audio_neucodec.zip:  98%|█████████████▊| 8.35MB / 8.48MB            

Processing Files (0 / 1)      :  98%|█████████████▊| 8.35MB / 8.48MB, 41.8MB/s  
New Data Upload               :  98%|█████████████▊| 8.35MB / 8.48MB, 41.8MB/s  

Processing Files (1 / 1)      : 100%|██████████████| 8.48MB / 8.48MB, 21.3MB/s  
New Data Upload               : 100%|██████████████| 8.48MB / 8.48MB, 21.3MB/s  

  ...Speech_audio_neucodec.zip: 100%|██████████████| 8.48MB / 8.48MB            

Processing Files (1 / 1)      : 100%|██████████████| 8.48MB / 8.48MB, 14.2MB/s  
New Data Upload               : 100%|██████████████| 8.48MB / 8.48MB, 14.2MB/s  
  ...Speech_audio_neucodec.zip: 100%|██████████████| 8.48MB / 8.48MB            
https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/blob/main/MsceneSpeech_audio_neucodec.zip


In [36]:
!hf upload malaysia-ai/MsceneSpeech MsceneSpeech_audio.zip --repo-type=dataset

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  MsceneSpeech_audio.zip      :   1%|▏             | 4.72MB /  490MB            

Processing Files (0 / 1)      :   1%|▏             | 4.72MB /  490MB, 23.5MB/s  
New Data Upload               :   7%|▉             | 4.72MB / 67.1MB, 23.5MB/s  

Processing Files (0 / 1)      :  28%|███▉          |  139MB /  490MB,  349MB/s  
New Data Upload               :  69%|█████████▋    |  139MB /  201MB,  349MB/s  

Processing Files (0 / 1)      :  55%|███████▋      |  270MB /  490MB,  450MB/s  
New Data Upload               :  80%|███████████▎  |  270MB /  335MB,  450MB/s  

Processing Files (0 / 1)      :  88%|████████████▎ |  431MB /  490MB,  539MB/s  
New Data Upload               :  88%|████████████▎ |  431MB /  490MB,  539MB/s  

Processing Files (0 / 1)      :  99%|█████████████▉|  488MB /  490MB,  487MB/s  
New Data Upload       